In [14]:
import torch
import torch.nn as nn

In [15]:
class Model(nn.Module): 

    def __init__(self, num_features): 
        super().__init__()
        self.linear = nn.Linear(in_features=num_features, out_features=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out
    


    

In [16]:
# create dataset
features = torch.rand(10, 5)

model = Model(num_features=features.shape[1])

# call model on features
model(features)

tensor([[0.6251],
        [0.5771],
        [0.5520],
        [0.6012],
        [0.6004],
        [0.5369],
        [0.5499],
        [0.6485],
        [0.6605],
        [0.5812]], grad_fn=<SigmoidBackward0>)

In [18]:
model.linear.weight, model.linear.bias

(Parameter containing:
 tensor([[0.1856, 0.4297, 0.1218, 0.3084, 0.1086]], requires_grad=True),
 Parameter containing:
 tensor([-0.1937], requires_grad=True))

In [20]:
from torchinfo import summary

summary(model, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [26]:
class Model2(nn.Module): 

    def __init__(self, num_features): 
        super().__init__()
        self.linear1 = nn.Linear(in_features=num_features, out_features=3)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(in_features=3, out_features=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear1(features)
        out = self.relu(out)
        out = self.linear2(out)
        out = self.sigmoid(out)
        return out
    


In [28]:
model = Model2(num_features=features.shape[1])
summary(model, input_size=(10, 5))


Layer (type:depth-idx)                   Output Shape              Param #
Model2                                   [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [29]:
model(features)

tensor([[0.3301],
        [0.3556],
        [0.3382],
        [0.3376],
        [0.3226],
        [0.3582],
        [0.3303],
        [0.3338],
        [0.3523],
        [0.3416]], grad_fn=<SigmoidBackward0>)

In [30]:
model.linear1.weight, model.linear1.bias

(Parameter containing:
 tensor([[ 0.3258, -0.1218,  0.3881, -0.4009, -0.2379],
         [ 0.3877,  0.4303, -0.4377, -0.0240, -0.0145],
         [ 0.2129, -0.2025, -0.0018, -0.1461,  0.1526]], requires_grad=True),
 Parameter containing:
 tensor([ 0.2523,  0.4404, -0.3016], requires_grad=True))

In [31]:
class Model3(nn.Module): 

    def __init__(self, num_features): 
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=num_features, out_features=3),
            nn.ReLU(),  
            nn.Linear(in_features=3, out_features=1),
            nn.Sigmoid()
        )

    def forward(self, features):
        out = self.network(features)
        return out

In [33]:
model = Model3(num_features=features.shape[1])
summary(model, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
Model3                                   [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

# Going back to Breast Cancer dataset

In [48]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [49]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [50]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [51]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['diagnosis']), df['diagnosis'], test_size=0.2, random_state=42)

In [52]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [53]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [63]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [ ]:
# Define the model

class MySimpleNN(nn.Module): 

    def __init__(self, num_features): 
        super().__init__()
        self.linear = nn.Linear(in_features=num_features, out_features=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out
    


In [72]:
learning_rate = 0.1
epochs = 25

In [78]:
loss_function = nn.BCELoss()
type(loss_function)

torch.nn.modules.loss.BCELoss

In [91]:
# Defining model 
model = MySimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loop

for epoch in range(epochs):

    # forward pass
    y_pred = model(X_train_tensor)

    # loss
    loss = loss_function(y_pred.squeeze(), y_train_tensor)

    # clear gradients
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameters update
    optimizer.step()
    

    # print loss in each epoch
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

Epoch 1/25, Loss: 0.4711669981479645
Epoch 2/25, Loss: 0.40030282735824585
Epoch 3/25, Loss: 0.35544681549072266
Epoch 4/25, Loss: 0.32411569356918335
Epoch 5/25, Loss: 0.300739586353302
Epoch 6/25, Loss: 0.28246405720710754
Epoch 7/25, Loss: 0.26767227053642273
Epoch 8/25, Loss: 0.25537756085395813
Epoch 9/25, Loss: 0.24494223296642303
Epoch 10/25, Loss: 0.23593461513519287
Epoch 11/25, Loss: 0.22805142402648926
Epoch 12/25, Loss: 0.22107276320457458
Epoch 13/25, Loss: 0.21483489871025085
Epoch 14/25, Loss: 0.20921316742897034
Epoch 15/25, Loss: 0.2041107565164566
Epoch 16/25, Loss: 0.1994510293006897
Epoch 17/25, Loss: 0.19517262279987335
Epoch 18/25, Loss: 0.1912255883216858
Epoch 19/25, Loss: 0.18756870925426483
Epoch 20/25, Loss: 0.18416784703731537
Epoch 21/25, Loss: 0.1809942126274109
Epoch 22/25, Loss: 0.17802339792251587
Epoch 23/25, Loss: 0.1752346009016037
Epoch 24/25, Loss: 0.1726098656654358
Epoch 25/25, Loss: 0.17013372480869293


In [92]:
# model evaluation
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f"Test Accuracy: {accuracy.item() * 100:.2f}%")



Test Accuracy: 52.80%
